### Problem Statement 3D

3D. Detect the faults in Sensor Data from Machinery (NASA Turbofan Engine Dataset) using Autoencoder and build the model by using the following steps:
Dataset: The dataset contains sensor readings from turbofan engines with labels indicating whether a reading is faulty.
Goal: Detect machinery faults by training an autoencoder on sensor data from healthy engines.
Steps:
- Preprocess data by scaling sensor readings.
- Use an LSTM-based autoencoder (to capture sequential data) to learn patterns in normal sensor readings.
- Calculate reconstruction error for test data, with a threshold to classify high-error instances as faults.

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Preprocess data by scaling sensor readings
csv_path = "../LP IV Updated Datasets/AutoEncoder/NASA Turbofan ENgine/NASA TurboFan Engine.csv"

data = pd.read_csv(csv_path)
print("Data shape:", data.shape)

# Select sensor columns that vary (drop constant sensor values)
sensor_cols = [c for c in data.columns if c.startswith('sensor') and data[c].std() > 0.01]
print("Selected sensor count:", len(sensor_cols))

# Scale sensor readings from 0 to 1
scaler = MinMaxScaler()
scaled_sensors = scaler.fit_transform(data[sensor_cols])

# Create sequential windows (length = 10 cycles)
TIME_STEPS = 10
X_seq = []
y_seq = []

for i in range(len(scaled_sensors) - TIME_STEPS):
    X_seq.append(scaled_sensors[i : i + TIME_STEPS])
    y_seq.append(data['Fault_Label'].values[i + TIME_STEPS])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Filter healthy engine data (Fault_Label == 0) for training
X_train_healthy = X_seq[y_seq == 0]

print("Healthy sequences for training:", X_train_healthy.shape[0])
print("Total sequences for testing:", X_seq.shape[0])

In [ ]:
# Use an LSTM-based autoencoder to learn patterns in normal sensor readings
n_features = len(sensor_cols)

model = Sequential()
# Encoder
model.add(LSTM(32, activation='relu', input_shape=(TIME_STEPS, n_features), return_sequences=False))
model.add(RepeatVector(TIME_STEPS))
# Decoder
model.add(LSTM(32, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(n_features)))

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Train LSTM autoencoder on healthy sensor readings
history = model.fit(X_train_healthy, X_train_healthy, epochs=5, batch_size=64, validation_split=0.1)

In [ ]:
# Calculate reconstruction error for test data and set fault threshold
predictions = model.predict(X_seq)

# Calculate MSE for each sequence
mse = np.mean(np.square(X_seq - predictions), axis=(1, 2))

# Set threshold at 95th percentile of healthy errors
threshold = np.percentile(mse[y_seq == 0], 95)
print("Machinery Fault Threshold:", threshold)

# Predict faults (error > threshold is a fault)
y_pred = (mse > threshold).astype(int)

from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_seq, y_pred, target_names=['Healthy', 'Fault']))

# Plot error progression
plt.figure(figsize=(10, 4))
plt.plot(mse[:300], label='Reconstruction Error')
plt.axhline(threshold, color='red', linestyle='--', label='Fault Threshold')
plt.title('Turbofan Machinery Sensor Reconstruction Error')
plt.xlabel('Cycle Index')
plt.ylabel('MSE')
plt.legend()
plt.show()